# RLSF reward-path smoke

---
## 1 — Setup

In [1]:
# 7B in bf16 is ~15 GB of weights; a T4 (16 GB) is tight, an A100/L4 is comfortable.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4090, 24564 MiB


In [2]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git pull
!git rev-parse --short HEAD

Cloning into 'Style-Aware-MT'...
remote: Enumerating objects: 1322, done.
remote: Counting objects: 100% (1322/1322), done.
remote: Compressing objects: 100% (557/557), done.
remote: Total 1322 (delta 879), reused 1196 (delta 753), pack-reused 0 (from 0)
Receiving objects: 100% (1322/1322), 15.58 MiB | 16.67 MiB/s, done.
Resolving deltas: 100% (879/879), done.
/workspace/Style-Aware-MT/notebooks/Style-Aware-MT
Already up to date.
4c2685e


In [3]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -r requirements-comet.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 78.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 100.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 40.8 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.1
    Uninstalling numpy-2.4.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/11 [numpy]
      Successfully uninstalled numpy-2.4.1━━━━━━━━━━━━━━━━━━━━  2/11 [numpy]
  Attempting uninstall: huggingface-hub0m━━━━━━━━━━━━━━━━━━━━━━━━━  4/11 [jsonargparse]
    Found existing installation: huggingface_hub 1.27.0━━━━━━━  4/11 [jsonargparse]
    Uninstalling huggingface_hub-1.27.0:━━━━━━━━━━━━━━━━━━━━━━  4/11 [jsonargparse]
      Successfully uninstalled h

In [5]:
import getpass, logging, os
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)

OPENAI_API_KEY:  ········


---
## 2 — Pre-flight


In [6]:
import hashlib, json, pathlib, yaml
from src.rlsf.config import load_config, reward_config
from src.rlsf.reward import load_train_template
from src.rlsf.smoke import plan

CONFIG   = 'configs/rlsf.yaml'
SEGMENTS = 20

cfg = load_config(CONFIG, require_caps=False)
G   = cfg['rlsf']['rollout']['group_size']

# -- the caps gate PPO training, not this pilot; if they are set, training was authorised
#    and this notebook is the wrong tool
assert all(cfg['rlsf']['caps'][k] is None for k in
           ('max_steps', 'max_grid_steps', 'max_judge_calls', 'max_judge_spend_usd')), \
    'training caps are declared; use the training runbook, not the smoke'

# -- the locked control: a quantized or swapped base is a different experiment
gen = cfg['generator']
assert gen['model'] == 'Qwen/Qwen2.5-7B-Instruct', gen['model']
assert gen['load_in_4bit'] is False, 'quantizing redefines the frozen base'
assert gen['adapter_path'], 'RLSF initializes from the frozen PEFT checkpoint'

# -- greedy rollouts give a group zero variance and the whole run is uninformative
assert cfg['rlsf']['rollout']['temperature'] > 0, 'greedy rollouts cannot be normalized'
assert G <= cfg['rlsf']['caps']['group_size_ceiling']

print(f"policy {gen['model']} + {gen['adapter_path']}")
print(f"rollout T={cfg['rlsf']['rollout']['temperature']} G={G}")
print('reward', reward_config(cfg))

policy Qwen/Qwen2.5-7B-Instruct + models/peft_lora_r32_lr2e-4/checkpoint-1358
rollout T=1.0 G=4
reward RewardConfig(w_bleu=1.0, w_kiwi=1.0, w_judge=1.0, len_min_ratio=0.5, len_max_ratio=2.0, on_violation='floor', overlap_metric='bleu')


In [7]:
# -- the reward judge must not be either evaluation rater, or training spends a rater on
#    the one condition that most needs a rater it was not trained against
raters = {yaml.safe_load(pathlib.Path(p).read_text())['judge']['model']
          for p in ('configs/judge_eval.yaml', 'configs/judge_eval_gpt.yaml')}
assert cfg['judge']['model'] not in raters, (cfg['judge']['model'], raters)

# -- seeded, because under group normalization a rater flipping a 3 to a 4 inverts an
#    advantage sign
assert cfg['judge']['temperature'] == 0.0 and cfg['judge']['seed'] == 42

# -- the rubric must be the frozen one; load_train_template raises on drift, this reports it
text = load_train_template()
digest = hashlib.sha256(text.encode()).hexdigest()
frozen = json.loads(pathlib.Path('prompts/hashes.json').read_text())['templates']
assert digest == frozen['judge_train.txt']['sha256']
assert cfg['template_file'] == 'prompts/judge_train.txt', 'the eval rubric would be circular'

print(f"reward judge {cfg['judge']['model']}, distinct from {sorted(raters)}")
print(f"rubric verified {digest[:16]}")

reward judge gpt-4o-mini, distinct from ['claude-haiku-4-5', 'gpt-5.6-terra']
rubric verified 8eaa11ff341c86ca


In [8]:
# -- the dev slice, against the manifest written when it was carved
man = json.loads(pathlib.Path('data/splits/rlsf_dev_manifest.json').read_text())
for name, want in man['hashes'].items():
    got = hashlib.sha256((pathlib.Path('data/splits') / name).read_bytes()).hexdigest()
    assert got == want, f'{name} differs from the manifest'
print(f"dev slice {man['counts']['rlsf_dev']} segments, {man['counts']['dev_works']} works")

# -- the slice is not unseen by the model; it selects weights, it does not measure them
print('\n'.join('  ' + c for c in man['caveats']))

p = plan(SEGMENTS, G)
print(f"\nplanned: {p['judge_calls']} judge calls, ~${p['est_usd']} "
      f"(pilot ceiling {cfg['rlsf']['pilot']['judge_calls']})")

dev slice 499 segments, 4 works
  Held out from PPO updates only. The PEFT checkpoint RLSF initializes from was trained on all of train.jsonl, this slice included, so the slice is not unseen by the model.
  results/stylometrics_centroid.json was built over all 10,860 train targets, including these works.
  Dev-slice figures select the reward weights and are never reported as a result; val remains the reported split.

planned: 80 judge calls, ~$0.0103 (pilot ceiling 80)


---
## 3 — Free pass

In [10]:
%%bash
set -e
df -h /workspace | tail -1
/usr/bin/python3 -m pip install --no-cache-dir -r requirements.txt
/usr/bin/python3 -c "import torch, transformers, trl, numpy; print('main ok', torch.__version__, torch.cuda.is_available(), transformers.__version__, trl.__version__)"

overlay          50G  4.6G   46G  10% /
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 67.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB

main ok 2.12.0+cu130 True 5.12.1 1.9.2


In [11]:
%%bash
set -e
df -h /workspace | tail -1
rm -rf .venv-comet
/usr/bin/python3 -m venv .venv-comet
.venv-comet/bin/pip install --no-cache-dir --upgrade pip
.venv-comet/bin/pip install --no-cache-dir "setuptools<81"
.venv-comet/bin/pip install --no-cache-dir -r requirements-comet.txt
.venv-comet/bin/python -c "import comet; print('comet ok')"
/usr/bin/python3 -c "import transformers; print('main still', transformers.__version__)"

overlay          50G   11G   40G  21% /
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.9 MB/s  0:00:0036m-:--:--
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 53.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 366.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 69.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 288.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 71.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━

/workspace/Style-Aware-MT/notebooks/Style-Aware-MT/.venv-comet/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


comet ok
main still 5.12.1


In [12]:
import getpass, os
os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

HF token:  ········


In [13]:
%%bash
.venv-comet/bin/python - <<'EOF'
import os
from huggingface_hub import HfApi, __version__
print('hub version:', __version__)
print('HF_TOKEN seen by worker env:', bool(os.environ.get('HF_TOKEN')))
api, tok = HfApi(), os.environ.get('HF_TOKEN')
try:
    print('whoami:', api.whoami(token=tok)['name'])
except Exception as e:
    print('TOKEN BAD:', type(e).__name__, str(e)[:200])
try:
    api.list_repo_files('Unbabel/wmt22-cometkiwi-da', token=tok)
    print('repo access: OK')
except Exception as e:
    print('REPO BLOCKED:', type(e).__name__, str(e)[:300])
EOF

hub version: 0.36.2
HF_TOKEN seen by worker env: True
whoami: prnamhr
repo access: OK


In [14]:
import sys

!{sys.executable} manage.py rlsf_smoke --config {CONFIG} --segments 4 --skip_judge \
    --out outputs/rlsf/smoke_free.jsonl

plan: 4 segments x G=4 = 16 samples
      judge skipped: 0 paid calls, judge component held flat
tokenizer_config.json: 7.30kB [00:00, 29.1MB/s]
vocab.json: 2.78MB [00:00, 8.32MB/s]
merges.txt: 1.67MB [00:00, 36.0MB/s]
tokenizer.json: 7.03MB [00:00, 30.8MB/s]
config.json: 100%|████████████████████████████| 663/663 [00:00<00:00, 1.53MB/s]
model.safetensors.index.json: 27.8kB [00:00, 49.6MB/s]
Fetching 4 files:   0%|                                  | 0/4 [00:00<?, ?it/s]
model-00001-of-00004.safetensors:   0%|            | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|            | 0.00/3.86G [00:00<?, ?B/s]


model-00004-of-00004.safetensors:   0%|            | 0.00/3.56G [00:00<?, ?B/s]



model-00003-of-00004.safetensors:   0%|            | 0.00/3.86G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   3%|    | 116M/3.86G [00:03<02:02, 30.7MB/s]
model-00001-of-00004.safetensors:   6%|▎   | 256M/3.95G [00:04<01:01, 59.8MB/s]

model-00002-of-00004.safetensors:   8

---
## 4 — Paid pass

In [15]:
!python3 manage.py rlsf_smoke --config {CONFIG} --segments {SEGMENTS} --group_size {G} --yes

plan: 20 segments x G=4 = 80 samples
      80 judge calls at concurrency 8, ~$0.0103 (ceiling 80)
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 114.74it/s]
sampling 4 completions per segment at T=1.0 ...
wrote outputs/rlsf/smoke_hyps.jsonl
kiwi worker ready (reference_free=True)

reward mean -0.121 sd 2.041, wrote outputs/rlsf/smoke_steps.jsonl
  bleu   degenerate groups 1/20
  kiwi   degenerate groups 1/20
  judge  degenerate groups 2/20
  reward degenerate groups 1/20  (min group sd 0.0)

drift stop: marker_rate z +0.577 +/- 0.295 over 20 prompts. The rule opens its baseline on the first 5 training steps and fires only after 3 consecutive steps past it.

  [PASS] kiwi handshake: worker ready
  [PASS] reward variance: 5% of groups have no reward spread across their feasible samples
  [PASS] steplog written: n_samples=80
  [PASS] length band: 75/80 feasible (94%), ratio mean 0.95, 0 unmeasured

judge usage: {'calls': 80, 'prompt_tokens': 33332, 'completion_tokens'

---
## 5 — Read the result


In [16]:
log = json.loads(pathlib.Path('outputs/rlsf/smoke_steps.jsonl').read_text().splitlines()[0])
print(f"samples {log['n_samples']}  feasible {log['n_feasible']} "
      f"({log['n_feasible'] / log['n_samples']:.0%})")
print(f"reward mean {log['reward_mean']:+.3f}  sd {log['reward_sd']:.3f}")
print(f"length mean {log['length_mean']:.1f} words, ratio to reference "
      f"{log['length_ratio_mean']:.2f}")
print('\nraw component means:', {k: round(v, 3) for k, v in log['raw'].items()})

# -- errors are prompt-clustered; nan marks a log written before z_se existed. They are
#    wide at 20 prompts, and the val-set z values in results/ are 1,323 greedy segments,
#    so these do not compare against that table -- the drift rule uses its own baseline
z_se = log.get('z_se') or dict.fromkeys(log['z'], float('nan'))
print('z-deviation from the register centroid:')
for k, v in log['z'].items():
    print(f"  {k:12s} {v:+.2f} +/- {z_se[k]:.2f}")

samples 80  feasible 75 (94%)
reward mean -0.121  sd 2.041
length mean 19.1 words, ratio to reference 0.95

raw component means: {'bleu': 25.229, 'kiwi': 0.664, 'judge': 3.175}
z-deviation from the register centroid:
  lex_density  -0.01 +/- 0.23
  ttr          +0.14 +/- 0.18
  root_ttr     -0.50 +/- 0.20
  marker_rate  +0.58 +/- 0.29


In [17]:
# The measured per-call rate. docs/budget.md carries an estimate over assumed token
# counts until this replaces it.
u = json.loads(pathlib.Path('outputs/rlsf/smoke_usage.json').read_text())
print(f"{u['calls']} calls, {u['prompt_tokens'] / u['calls']:.0f} in / "
      f"{u['completion_tokens'] / u['calls']:.0f} out per call")
print(f"measured ${u['per_call_usd']:.6f}/call against the $0.000114 assumed")
print(f"\nprojected at the 44,800-call ceiling: ${u['per_call_usd'] * 44_800:.2f} "
      f"(docs/budget.md records $5.11-$6.45)")

80 calls, 417 in / 18 out per call
measured $0.000074/call against the $0.000114 assumed

projected at the 44,800-call ceiling: $3.30 (docs/budget.md records $5.11-$6.45)
